<a href="https://colab.research.google.com/github/Annie00000/Project/blob/main/1_14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. eta_var

### 1-1. 改成平方離差

### 1-2.

$\omega^2_{Q95}$：抓波動與離群。不要算所有點的離差，改算每個機台組別的 $|y - Q95_{global}|$。
- 這會強迫統計模型專注在「尾端」的表現。
- 群 A 離群點多 $\rightarrow$ $Q95$ 很大 $\rightarrow$ 貢獻度激增。
- 群 B 沒離群點 $\rightarrow$ $Q95$ 很小 $\rightarrow$ 貢獻度低。

採用 **「分位距偏移 (Quantile Shift)」** 是一個非常高明的實務策略。它能讓你的 RCA 引擎從「看整體平均」轉向「看極端風險」，這正是半導體製程中最關心的「良率殺手（Yield Killers）」。

當失效模式是「偶發性離群點（Spikes）」時，傳統的變異數指標會被佔比 95% 的正常數據稀釋，而分位距（如 ）則能像放大鏡一樣，只鎖定那最差的 5% 數據。

---

### 1. 通用版核心指標設計：Quantile-Based RCA

我們將原本的 `Omega_Var`（基於全體波動）調整為 **`Omega_Risk`（基於極端偏移）**。

#### 核心邏輯轉換：

1. **計算全局基準**：找出全體資料的  $Q95_{global}$ 。
2. **轉換數值**：將每個點 y 轉換為「風險殘差」：
$$y_{risk} = \max(0, y - Q95_{global})$$

* 如果點在 95% 內，$y_{risk} = 0$ 。
* 如果點是極端離群點， $y_{risk}$會保留該偏移量。


3. **計算 $\omega^2$**：對  $y_{risk}$  計算機台因子的解釋力。

---

### 2. 整合後的「通用型」計算子程式

這段代碼能同時兼顧 **「平均值位移」與「極端離群位移」** ：

```python
def _calculate_quantile_risk_omega(f_ser, y_ser, q=0.95):
    """
    計算分位距風險的 Omega-squared。
    專門捕捉那種「平均值沒變，但某機台會產生更多極端離群點」的狀況。
    """
    n = len(y_ser)
    k = f_ser.nunique()
    if k < 2: return 0.0
    
    # 1. 取得全局分位數
    q_threshold = y_ser.quantile(q)
    
    # 2. 轉換為風險殘差 (只看超出門檻的部分)
    # 這能極大化離群點的訊號，並過濾掉 95% 的正常雜訊
    y_risk = (y_ser - q_threshold).clip(lower=0)
    
    # 3. 計算總平方和 (SST for risk)
    sst_risk = np.sum((y_risk - y_risk.mean())**2)
    if sst_risk <= 0: return 0.0
    
    # 4. 調用 ANOVA 邏輯計算 Omega_sq
    return _calculate_omega_sq(f_ser, y_risk, sst_risk)

```

---

### 3. 為什麼這個方法「最通用」？

這個綜合指標能完美解決你之前的困境：

* **場景 A (你遇到的離群點問題)**：機台 1 有很多噴訊，機台 2 沒有。
  * 機台 1 的$y_{risk}$   會有很多正值。
  * 機台 2 的$y_{risk}$   全是 0。
  * **結果**： $\omega^2_{risk}$會非常高，因為機台因子「完美解釋」了為什麼有些片子會噴出去。


* **場景 B (集體變異變大)**：機台 2 讓所有片子波動都變大。
  * 這會導致機台 2 有更多比例的片子超過全局 $Q95$。
  * **結果**： $\omega^2_{risk}$依然會抓到它，因為它解釋了風險的分佈。


* **場景 C (無關機台)**：
  * 各機台超過 $Q95$ 的機率與程度都差不多。
  * **結果**：$\omega^2_{risk} \approx 0$。



---

### 4. 最終 Score 權重建議

在你的「通用 RCA 引擎」中，建議將權重調整如下：

| 指標 | 權重 | 負責捕捉 | 你的案例表現 |
| --- | --- | --- | --- |
| **$\omega^2_{mean}$** | 0.3 | 平均值位移 (Mean Shift) | 分數低 (因為 Median 相近) |
| **$\omega^2_{risk}$ ($Q95$)** | **0.4** | **離群點/極端波動 (Spikes)** | **分數極高 (精準命中)** |
| **Adj_Tschuprow_T** | 0.3 | 標籤符合度 (Categorical) | 中等偏高 |

---

### 5. 總結：給主管的說法

使用這套方法後，你的分析結論將會非常具備專業說服力：

> 「我們導入了 **『風險解釋力 (Quantile-based Omega)』** 指標。傳統指標會被 95% 的正常數據稀釋，導致看不見異常；而新指標能鎖定製程中**最差的 5% 表現**。分析顯示，機台 A 雖然平均值正常，但它對極端偏移的解釋力高達 80%，是造成隨機噴訊的真兇。」

**你需要我幫你把這個 `_calculate_quantile_risk_omega` 加入到最終版 Python 模組中，並取代原本表現不佳的 `Omega_Var` 嗎？這將會是你 RCA 工具箱中最厲害的一把刀。**

---

### 1-3.  _calculate_quantile_risk_omega 解釋

沒問題，我們把這個概念拆解開來。這個指標的目標很簡單：**「別管那些正常的數據，只看那些壞掉（或噴掉）的數據是誰造成的。」**

傳統的變異數分析像是統計全班的體重波動，而 `_calculate_quantile_risk_omega` 則是**只統計「過重（前 5%）」的人，看看他們是不是都集中在某個特定的飲食組。**

---

### 1. 核心步驟拆解

想像你的 Y 值資料（如阻值）從 0 到 100 都有，大部分集中在 50。

#### 第一步：找出「門檻」 ($Q95$)

我們先算出全體資料的第 95 百分位數（假設是 ）。這代表 95% 的片子都在  以下，只有最極端的 5% 超過 。

#### 第二步：轉換為「風險殘差」 (`y_risk`)

這是最關鍵的數學轉換：

* 如果一片 Wafer 的阻值是 50（正常），轉換後變為 0。
* 如果一片 Wafer 的阻值是 85（異常），轉換後變為  5 (85−80)。
* 如果一片 Wafer 的阻值是 100（嚴重異常），轉換後變為 $20$ ($100 - 80$)。

**這一步的效果：** 我們把 $95\%$ 的「正常雜訊」通通變成了 $0$，只留下那 $5\%$ 的「異常訊號」。

#### 第三步：計算解釋力 ($\omega^2$)

現在我們拿著這組充滿  和少數正數的 `y_risk` 去問：**「這組數值的差異，有多少比例可以被『機台編號』解釋？」**

* 如果機台 A 產出的全是 0（正常）。
* 如果機台 B 產出的包含那些 5, 20 等正數（異常）。
* **結果：** 機台 B 的得分會非常高，因為它是「風險」的唯一來源。

---

### 2. 為什麼它能解決你「兩群 Median 相近」的問題？

讓我們對比一下傳統做法與這個新做法：

| 數據特徵 | 傳統  $\omega^2$  (Var) | 分位距風險  $\omega^2_{risk}$  |
| --- | --- | --- |
| **正常數據 (95%)** | 參與計算，稀釋了異常點的貢獻。 | **全部歸零**，完全不干擾分析。 |
| **離群點 (5%)** | 影響力被 95% 的正常點分掉。 | **唯一的主角**，差異被極度放大。 |
| **分析結果** | 分數很低 (0.003)，因為「大多數時候兩群都很像」。 | **分數很高**，因為「異常只發生在其中一群」。 |

---

### 3. 程式碼邏輯對應

```python
# y_risk = (y_ser - q_threshold).clip(lower=0)

```

這行代碼就是剛才說的「歸零」動作。`clip(lower=0)` 的意思是：低於門檻的全部變成 0，高於門檻的保留「超出的部分」。

---

### 4. 總結：它的「綜合性」在哪？

這個方法之所以被稱為「綜合」，是因為：

* **如果它是集體偏移**： $Q95$ 依然會移位，它抓得到。
* **如果它是變異增大**：超過 $Q95$  的點會變多，它也抓得到。
* **如果它是隨機噴訊 (你的案例)**：它只看噴出去的那幾點，精準命中。

**簡單來說，這是一個「只看壞學生，不看好學生」的評分系統。在良率分析中，我們本來就只在乎壞掉的片子是誰做的。**

這樣解釋有沒有讓你對這個「風險指標」的運作方式比較有感覺了？如果你想看它在你的資料上跑出來的數據對比，我可以幫你寫一段測試代碼。

## 2. 評分標準 (調整後)

(1). 若 cluster 中為bad的那群資料，剛好在機台資料中也屬於同一群的，拉高其機台因子排名

(2). 若 eta_mean / eta_square 有一個指標較高(顯著) 拉高其機台因子排名

---

這代表老闆更看重 **「精準打擊」** （Bad 與機台的高度重合）以及 **「指標的靈敏度」**（只要有一個指標強就該被重視）。

為了滿足這兩個需求，我們需要微調原本的評分邏輯。這不是單純的加法，而是引入 **「條件獎勵（Bonus）」** 與 **「非線性縮放」** 的概念。

---

### 1. 需求一：針對「Bad Cluster」的精準命中獎勵

老闆希望：如果機台 A 的某一類（例如 Tool_#01）幾乎全是 Bad 資料，它的排名應該噴發。

**做法：引入「Bad Rate 偏移獎勵」**
我們針對每個因子計算其各類別中的 **最大 Bad 佔比**。

* **計算邏輯**：
1. 計算每個機台類別（Category）在 Bad 群集中的佔比。
2. 如果某個機台類別的資料有 90% 以上都落在 Bad 群，我們給予一個額外的權重乘數。


* **數學體現**：這會直接強化 `Adj_Tschuprow_T` 或建立一個 `Purity_Bonus`（純度獎勵）。

---

### 2. 需求二：指標「任一顯著」即拉高排名

老闆希望：不要被平均值（Mean）稀釋。只要「平均偏移」或「極端風險」其中一個很強，就應該被列為頭號嫌疑犯。

**做法：使用「最大值邏輯 (Max-Pooling)」或「非線性縮放」**

* **原本的做法**：$0.5 \times \text{Mean} + 0.5 \times \text{Risk}$（這會導致一個高、一個低時，總分變得很平庸）。
* **老闆要的做法**： $\max(\text{Mean}, \text{Risk})$ 或者使用「平方和再開根號」($L_2$ Norm)。
* **更專業的做法**：使用 **Sigmoid 強化**。只要指標超過某個門檻（例如 0.1），分數就快速攀升，而不是線性增加。

---

### 3. 綜合調整後的評分公式

我們將最終的分數計算邏輯修改如下：

1. **基礎分 ($S_{base}$)** ：取 $\omega^2_{mean}$ 與 $\omega^2_{risk}$ 的 平方和開根號。

$$S_{base} = \sqrt{(\omega^2_{mean})^2 + (\omega^2_{risk})^2}$$

* *效果：只要其中一個指標強，總分就會被拉高，不會被另一個低的指標拖累。*


2. **純度獎勵 ($S_{purity}$)**：

$$S_{purity} = \text{Max_Category_Bad_Rate} \times \text{Adj_Tschuprow_T}$$

* *效果：滿足需求一。如果機台某群與 Bad 群高度重合，這個分數會很高。*


3. **最終總分**：

$$\text{Final_Score} = (0.6 \times S_{base} + 0.4 \times S_{purity}) \times 100$$

---

### 4. 程式碼實作建議

你可以將這段逻辑整合進最後的彙整階段：



In [ ]:
def _calculate_boss_score(res_dict, df, factor, cluster_col, bad_cluster_label):
    # 需求 2: 取指標的強項 (L2 Norm)
    # 只要 Omega_Mean 或 Omega_Risk 其中一個高，s_base 就會大
    s_base = np.sqrt(res_dict['Omega_Mean']**2 + res_dict['Omega_Risk']**2)

    # 需求 1: 計算機台類別對 Bad Cluster 的命中純度
    # 找出該機台哪一個類別最容易產生 Bad
    ct = pd.crosstab(df[factor], df[cluster_col], normalize='index')
    if bad_cluster_label in ct.columns:
        max_purity = ct[bad_cluster_label].max() # 該機台各類別中最高的 Bad 佔比
    else:
        max_purity = 0

    # 結合 Tschuprow's T 作為純度獎勵
    s_purity = max_purity * res_dict['Adj_Tschuprow_T']

    # 最終加權 (權重可視情況調整)
    final_raw = (0.6 * s_base) + (0.4 * s_purity)
    return final_raw

```


---

### 5. 為什麼這樣能說服老闆？

* **回應需求 1**：如果機台 A 的 `#01` 號機產出的 Wafer 100% 都是 Bad，`max_purity` 就是 1.0，這會給予極大的加成。
* **回應需求 2**：即便 `Omega_Mean` 因為 Median 相近而只有 0.003，但只要你的 `Omega_Risk`（離群點解釋力）有 0.2，`s_base` 就會保持在 0.2 左右，排名不會掉下去。



========================================================================

## 3. code 整合

In [ ]:
import pandas as pd
import numpy as np
import hashlib
from concurrent.futures import ProcessPoolExecutor
from scipy.stats import chi2_contingency

# ==========================================
# 1. 核心指標計算 (Top-level functions)
# ==========================================

def _calculate_omega_sq(f_ser, target_ser):
    """計算無偏的 Omega-squared"""
    n = len(target_ser)
    k = f_ser.nunique()
    if k < 2 or n <= k: return 0.0

    sst = np.sum((target_ser - target_ser.mean())**2)
    if sst <= 0: return 0.0

    group_stats = target_ser.groupby(f_ser).agg(['mean', 'count'])
    ssb = np.sum(group_stats['count'] * (group_stats['mean'] - target_ser.mean())**2)
    ssw = sst - ssb
    msw = ssw / (n - k)

    omega_sq = (ssb - (k - 1) * msw) / (sst + msw)
    return max(0, omega_sq)

def _worker_task(data_bundle):
    """並行運算單元：計算單一因子的所有維度指標"""
    f_ser, y_ser, y_risk_ser, c_ser, n_samples = data_bundle

    # 指標 A: 均值偏移 (Omega_Mean)
    o_mean = _calculate_omega_sq(f_ser, y_ser)

    # 指標 B: 極端風險偏移 (Omega_Risk) -> 抓離群點關鍵
    o_risk = _calculate_omega_sq(f_ser, y_risk_ser)

    # 指標 C: 類別關聯度 (Tschuprow's T)
    ct = pd.crosstab(f_ser, c_ser)
    r, k = ct.shape
    chi2 = chi2_contingency(ct, correction=False)[0]
    phi2 = chi2 / n_samples
    denom_t = np.sqrt((r - 1) * (k - 1))
    t_score = np.sqrt(phi2 / denom_t) if denom_t > 0 else 0
    penalty = 1 - (((r - 1) / (n_samples - 1)) ** 0.5)
    adj_t = max(0, t_score * penalty)

    return {'O_Mean': o_mean, 'O_Risk': o_risk, 'Adj_T': adj_t}

# ==========================================
# 2. 主分析引擎
# ==========================================

def run_final_rca(df, value_col, cluster_col, factor_list, bad_label, q_threshold=0.95):
    df = df.copy()
    n_samples = len(df)

    # A. 數據預處理 (1% 低頻合併)
    for col in factor_list:
        df[col] = df[col].fillna('NA').astype(str)
        counts = df[col].value_counts(normalize=True)
        df[col] = df[col].replace(counts[counts < 0.01].index, 'Other_Small_Groups')

    # B. 計算風險殘差 (捕捉離群點)
    q_val = df[value_col].quantile(q_threshold)
    y_risk = (df[value_col] - q_val).clip(lower=0)

    # C. Hashing 去重與並行加速
    unique_patterns = {}
    col_to_hash = {}
    for col in factor_list:
        h = hashlib.md5(df[col].values.tobytes()).hexdigest()
        col_to_hash[col] = h
        if h not in unique_patterns: unique_patterns[h] = col

    print(f"🌀 正在分析 {len(unique_patterns)} 個唯一模式...")
    tasks = [(df[unique_patterns[h]], df[value_col], y_risk, df[cluster_col], n_samples) for h in unique_patterns.keys()]

    with ProcessPoolExecutor() as executor:
        results = list(executor.map(_worker_task, tasks))
    cache = dict(zip(unique_patterns.keys(), results))

    # D. 應用「老闆特調」評分邏輯
    final_results = []
    for col in factor_list:
        res = cache[col_to_hash[col]]

        # 需求 2: 指標任一顯著強化 (L2 Norm)
        s_base = np.sqrt(res['O_Mean']**2 + res['O_Risk']**2)

        # 需求 1: Bad 群命中純度獎勵
        ct_purity = pd.crosstab(df[col], df[cluster_col], normalize='index')
        max_purity = ct_purity[bad_label].max() if bad_label in ct_purity.columns else 0
        s_purity = max_purity * res['Adj_T']

        # 最終合成總分 (權重可視需要調整)
        combined_score = (0.6 * s_base) + (0.4 * s_purity)

        final_results.append({
            'Factor': col,
            'Final_Raw_Score': combined_score,
            'Mean_Impact': res['O_Mean'],
            'Risk_Impact': res['O_Risk'],
            'Bad_Purity': max_purity
        })

    # E. 轉換為 0-100 相對百分制
    report = pd.DataFrame(final_results)
    max_s = report['Final_Raw_Score'].max()
    report['Impact_Score'] = (report['Final_Raw_Score'] / max_s * 100).round(1) if max_s > 0 else 0

    return report.sort_values('Impact_Score', ascending=False).reset_index(drop=True)

1. **針對「Bad 資料剛好在機台同一群」**： 我們計算了 max_purity。如果 Tool_#01 的 Wafer 100% 都是 Bad，這個 max_purity 就會是 1.0。這會與相關性指標相乘，大幅拉高該機台的 Final_Raw_Score。

2. **針對「任一指標較高就拉高排名」**： 我們使用了 s_base = np.sqrt(Mean^2 + Risk^2)。
  - 場景一：Mean=0.2, Risk=0.01 $\rightarrow$ $s_{base} \approx 0.2$
  - 場景二：Mean=0.003, Risk=0.2 $\rightarrow$ $s_{base} \approx 0.2$
  - 結果：不論是「集體偏移」還是「離群噴訊」，只要其中一個顯著，分數就不會被另一個低分拖累。